# Task 3: Domain Generalization (PACS) — GPU Colab (clone workflow)

Use this when the **code Drive account is out of GPU quota**.

1. Sign into Colab with the **GPU account** (account B).
2. Runtime → GPU.
3. This notebook **clones GitHub** (does not mount the code Drive).
4. You must place Task 2 ERM checkpoint `source_only_best.pt` (gitignored) once.

Checkpoint selection = mean source-val macro-F1 only. Sketch labels only in final eval.

In [ ]:
import torch

print("cuda:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("No GPU — Runtime → Change runtime type → GPU, then re-run.")
print("device:", torch.cuda.get_device_name(0))

## Clone repo from GitHub

Public clone into `/content/ATML-PA1` (fast local disk). Optional: mount **this** account's Drive only to **save results** at the end.

In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/ttqureshi/ATML-PA1.git"
REPO_DIR = Path("/content/ATML-PA1")

if (REPO_DIR / ".git").is_dir():
    print("Repo exists — pulling latest main...")
    subprocess.check_call(["git", "-C", str(REPO_DIR), "fetch", "origin"])
    subprocess.check_call(["git", "-C", str(REPO_DIR), "checkout", "main"])
    subprocess.check_call(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", "main"])
else:
    print("Cloning", REPO_URL)
    subprocess.check_call(["git", "clone", "--branch", "main", REPO_URL, str(REPO_DIR)])

os.chdir(REPO_DIR)
print("cwd:", Path.cwd())
print("task3:", (REPO_DIR / "task3").is_dir())

In [ ]:
%pip install -q -r requirements.txt gdown

## Place ERM checkpoint (required once)

Target path:

`task2/results/checkpoints/source_only_best.pt`

Pick **one** method below (A upload / B gdown / C local path). Do not retrain Source-only.

In [ ]:
# === Method A: upload from the PC that has account-A Drive synced ===
# Uncomment to use:
#
# from google.colab import files
# from pathlib import Path
# import shutil
#
# dest = Path("task2/results/checkpoints")
# dest.mkdir(parents=True, exist_ok=True)
# uploaded = files.upload()  # choose source_only_best.pt
# for name in uploaded:
#     shutil.move(name, dest / "source_only_best.pt")
# print("Saved", dest / "source_only_best.pt")

print("Method A is commented out — use B or C, or uncomment A.")

In [ ]:
# === Method B: gdown from a Drive file link shared "Anyone with the link" ===
# On account A: right-click source_only_best.pt → Share → Anyone with link → copy file id.
# File id is the part after /d/ in:
#   https://drive.google.com/file/d/<FILE_ID>/view

from pathlib import Path
import gdown

ERM_GDRIVE_FILE_ID = ""  # <-- paste file id here

dest = Path("task2/results/checkpoints/source_only_best.pt")
dest.parent.mkdir(parents=True, exist_ok=True)

if dest.exists() and dest.stat().st_size > 1_000_000:
    print("ERM ckpt already present:", dest, dest.stat().st_size, "bytes")
elif not ERM_GDRIVE_FILE_ID.strip():
    print("Set ERM_GDRIVE_FILE_ID, or use Method A/C.")
else:
    url = f"https://drive.google.com/uc?id={ERM_GDRIVE_FILE_ID}"
    gdown.download(url, str(dest), quiet=False)
    print("Downloaded", dest, dest.stat().st_size, "bytes")

In [ ]:
# === Method C: copy from a path already on this Colab / B's Drive ===
# Example after mounting B Drive and uploading the file there:
# SRC = "/content/drive/MyDrive/source_only_best.pt"

from pathlib import Path
import shutil

SRC = ""  # <-- absolute path to the .pt on this runtime, or leave empty
dest = Path("task2/results/checkpoints/source_only_best.pt")

if dest.exists() and dest.stat().st_size > 1_000_000:
    print("ERM ckpt already present:", dest.stat().st_size, "bytes")
elif not SRC.strip():
    print("SRC empty — skip Method C.")
else:
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(SRC, dest)
    print("Copied ->", dest, dest.stat().st_size, "bytes")

In [ ]:
!python -m task3.scripts.run_task3 --stages check_erm

## PACS download + splits

In [ ]:
from pathlib import Path
import shutil
import zipfile
import urllib.request

pacs_root = Path("data/pacs")

def _count(d):
    return sum(1 for p in (pacs_root / d).rglob("*") if p.is_file())

if (pacs_root / "photo").is_dir() and _count("photo") > 0:
    print("PACS already present at", pacs_root.resolve())
else:
    pacs_root.parent.mkdir(parents=True, exist_ok=True)
    zip_path = Path("data/PACS.zip")
    if not zip_path.exists():
        url = "https://huggingface.co/datasets/Azeez577/PACS/resolve/main/PACS.zip"
        print("Downloading", url)
        urllib.request.urlretrieve(url, zip_path)
    extract_dir = Path("data/_pacs_extract")
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    extract_dir.mkdir(parents=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_dir)
    candidates = [c.parent for c in extract_dir.rglob("photo") if c.is_dir()]
    if not candidates:
        raise FileNotFoundError("Could not locate PACS photo/ after unzip")
    src = candidates[0]
    if pacs_root.exists():
        shutil.rmtree(pacs_root)
    shutil.move(str(src), str(pacs_root))
    print("Installed PACS ->", pacs_root.resolve())

for d in ["photo", "art_painting", "cartoon", "sketch"]:
    print(f"  {d}: {_count(d)} files")

!python -m shared.prepare_pacs_splits

## Train main (DAN-DG λ=1, SAM ρ=0.05)

ERM is load-only.

In [ ]:
!python -m task3.scripts.run_task3 --stages train_main

## Controlled study λ_DG ∈ {0.1, 1, 10}

Re-runs λ=1 (same as main) for a consistent study folder naming; safe to keep.

In [ ]:
!python -m task3.scripts.run_task3 --stages study_lambda

## Source diagnostics (no Sketch) → final Sketch eval

In [ ]:
!python -m task3.scripts.run_task3 --stages eval_source,eval

## Copy results back to account-A Drive (optional)

Mount **this** (GPU) account's Drive, or download the zip to your PC and drop into account-A repo.

In [ ]:
from pathlib import Path
import shutil

src = Path("/content/ATML-PA1/task3/results")
zip_path = Path("/content/task3_results_bundle")
shutil.make_archive(str(zip_path), "zip", root_dir=src)
print("Created", str(zip_path) + ".zip")
print("Download it from the Colab file browser, then unzip into account-A:")
print("  .../ATML-PA1/task3/results/")

# Optional: also save onto GPU-account Drive
# from google.colab import drive
# drive.mount("/content/drive")
# out = Path("/content/drive/MyDrive/task3_results_bundle.zip")
# shutil.copy2(str(zip_path) + ".zip", out)
# print("Also copied to", out)